# Deliverable 5 — Great Expectations Quality Gate and OpenLineage

## Project Objective

This notebook implements the quality-gate and lineage requirements for the **Food Delivery Analytics Lakehouse** created in Deliverable 2.

The implementation:

1. Reads the persisted **Silver Delta table** from Deliverable 2
2. Validates Silver data using **Great Expectations**
3. Raises an exception when the quality gate fails
4. Demonstrates both the successful and failing paths
5. Emits real OpenLineage `START`, `COMPLETE`, and `FAIL` events
6. Writes lineage events to a local JSON log file for inspection

## Integration with Deliverable 2

Deliverable 2 writes its Silver Delta table to:

```text
/content/drive/MyDrive/food_delivery_capstone/delivery_data/silver
```

A separate notebook cannot access the temporary Python variable `silver_df`. This notebook therefore reads the persisted Delta table directly.

When Deliverables 2 and 5 run in different Colab sessions, use the shared Google Drive path option described below.

## 1. Environment Setup

The PySpark and Delta Lake versions match Deliverable 2:

- `pyspark==3.5.0`
- `delta-spark==3.2.0`

Great Expectations performs the quality checks, while OpenLineage emits structured lineage events.

In [1]:
# Install the real libraries required by Deliverable 5.
# PySpark and Delta Lake versions intentionally match Deliverable 2.

!pip install -q \
    pyspark==3.5.0 \
    delta-spark==3.2.0 \
    great-expectations \
    openlineage-python

## 2. Imports and Spark Configuration

The Spark session is configured with Delta Lake extensions so it can read the Silver table produced by Deliverable 2.

In [2]:
import json
import os
from datetime import UTC, datetime
from pathlib import Path
from typing import Dict, List, Optional

import great_expectations as gx
import great_expectations.expectations as gxe

from delta import configure_spark_with_delta_pip
from pyspark.sql import DataFrame, SparkSession

from openlineage.client import OpenLineageClient
from openlineage.client.event_v2 import (
    Dataset,
    Job,
    Run,
    RunEvent,
    RunState,
)
from openlineage.client.transport.file import (
    FileConfig,
    FileTransport,
)
from openlineage.client.uuid import generate_new_uuid


def create_spark_session() -> SparkSession:
    """Create a local Spark session configured for Delta Lake."""

    builder = (
        SparkSession.builder
        .appName("FoodDeliveryQualityGate")
        .master("local[*]")
        .config(
            "spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension"
        )
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog"
        )
    )

    return configure_spark_with_delta_pip(
        builder
    ).getOrCreate()


spark = create_spark_session()
spark.sparkContext.setLogLevel("ERROR")

print("Spark session created successfully.")
print("Spark version:", spark.version)

Spark session created successfully.
Spark version: 3.5.0


## 3. Shared Silver Delta Table Location

Deliverable 2 writes its Delta tables to Google Drive. This notebook mounts the same Drive and reads the persisted Silver table from:

```text
/content/drive/MyDrive/food_delivery_capstone/delivery_data/silver
```

Because the path is persistent, Deliverable 5 remains a separate notebook and does not depend on another Colab tab sharing the same temporary runtime.

In [3]:
from google.colab import drive

# Mount the same persistent storage used by Deliverable 2.
drive.mount("/content/drive")

BASE_PATH = (
    "/content/drive/MyDrive/"
    "food_delivery_capstone/delivery_data"
)
SILVER_PATH = f"{BASE_PATH}/silver"

print("Configured Silver Delta path:", SILVER_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Configured Silver Delta path: /content/drive/MyDrive/food_delivery_capstone/delivery_data/silver


## 4. OpenLineage Configuration

OpenLineage records the execution state of each stage.

This notebook emits:

- `START` before a stage begins
- `COMPLETE` after successful completion
- `FAIL` when an exception or quality failure occurs

Events are written to:

```text
lineage_events/openlineage.log
```

In [4]:
# Create the lineage output directory.
LINEAGE_DIRECTORY = Path("lineage_events")
LINEAGE_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

LINEAGE_LOG_PATH = (
    LINEAGE_DIRECTORY / "openlineage.log"
)

# Start with a clean lineage log for this demonstration.
if LINEAGE_LOG_PATH.exists():
    LINEAGE_LOG_PATH.unlink()

lineage_client = OpenLineageClient(
    transport=FileTransport(
        FileConfig(
            log_file_path=str(LINEAGE_LOG_PATH)
        )
    )
)

LINEAGE_NAMESPACE = "food-delivery-capstone"


def make_dataset(
    name: str,
    namespace: str = "local-delta"
) -> Dataset:
    """Create an OpenLineage dataset reference."""

    return Dataset(
        namespace=namespace,
        name=name
    )


def emit_lineage_event(
    job_name: str,
    run_id: str,
    state: RunState,
    inputs: Optional[List[Dataset]] = None,
    outputs: Optional[List[Dataset]] = None
) -> None:
    """Emit one OpenLineage event for a pipeline stage."""

    lineage_client.emit(
        RunEvent(
            eventType=state,
            eventTime=datetime.now(
                UTC
            ).isoformat(),
            run=Run(
                runId=run_id
            ),
            job=Job(
                namespace=LINEAGE_NAMESPACE,
                name=job_name
            ),
            producer="https://openlineage.io",
            inputs=inputs or [],
            outputs=outputs or []
        )
    )

    print(
        f"OpenLineage event emitted: "
        f"{job_name} → {state}"
    )


silver_dataset = make_dataset(
    SILVER_PATH
)

## 5. Load the Silver Delta Table

This stage reads the persisted Silver table rather than referencing `silver_df` from Deliverable 2.

If the path does not exist, the notebook gives a clear integration error instead of producing a `NameError`.

In [5]:
def load_silver_delta_table(
    silver_path: str
) -> DataFrame:
    """Read the Silver Delta table created by Deliverable 2."""

    if not os.path.exists(silver_path):
        raise FileNotFoundError(
            "Silver Delta table was not found at: "
            f"{silver_path}\n\n"
            "Run Deliverable 2 first so it writes the Silver table "
            "to the shared Google Drive path."
        )

    return (
        spark.read
        .format("delta")
        .load(silver_path)
    )


load_run_id = str(
    generate_new_uuid()
)

emit_lineage_event(
    job_name="load_silver_delta",
    run_id=load_run_id,
    state=RunState.START,
    inputs=[silver_dataset]
)

try:
    silver_df = load_silver_delta_table(
        SILVER_PATH
    )

    print("Silver Delta table loaded successfully.")
    print("Silver row count:", silver_df.count())
    silver_df.orderBy("delivery_id").show(
        truncate=False
    )

    emit_lineage_event(
        job_name="load_silver_delta",
        run_id=load_run_id,
        state=RunState.COMPLETE,
        inputs=[silver_dataset]
    )

except Exception:
    emit_lineage_event(
        job_name="load_silver_delta",
        run_id=load_run_id,
        state=RunState.FAIL,
        inputs=[silver_dataset]
    )

    raise

OpenLineage event emitted: load_silver_delta → EventType.START
Silver Delta table loaded successfully.
Silver row count: 7
+-----------+-----------+---------------+------------+----------------+---------+----------------+
|delivery_id|customer_id|restaurant_name|order_amount|delivery_minutes|city_zone|delivery_status |
+-----------+-----------+---------------+------------+----------------+---------+----------------+
|DEL001     |CUS001     |Pizza Palace   |85.5        |28              |NORTH    |completed       |
|DEL002     |CUS002     |Burger House   |42.0        |18              |EAST     |completed       |
|DEL003     |CUS003     |Sushi Corner   |110.25      |40              |WEST     |completed       |
|DEL004     |CUS004     |Pasta Villa    |63.75       |25              |SOUTH    |completed       |
|DEL005     |CUS005     |Taco Spot      |38.5        |20              |NORTH    |out for delivery|
|DEL006     |CUS006     |Burger House   |55.0        |22              |EAST     |comp

## 6. Great Expectations Validation Suite

The validation suite checks important data-quality dimensions:

| Quality dimension | Expectation |
|---|---|
| Completeness | `delivery_id` is not null |
| Uniqueness | `delivery_id` is unique |
| Completeness | `customer_id` is not null |
| Validity | `order_amount` is at least 0.01 |
| Validity | `delivery_minutes` is at least 1 |
| Validity | `city_zone` belongs to the approved domain |
| Validity | `delivery_status` belongs to the approved domain |

These rules are aligned with the Silver schema and transformed values in Deliverable 2.

In [6]:
def run_great_expectations_checkpoint(
    dataframe
):
    """Validate the Silver dataset with Great Expectations."""

    # Use an ephemeral context so the notebook is self-contained.
    context = gx.get_context(
        mode="ephemeral"
    )

    data_source = context.data_sources.add_pandas(
        "food_delivery_data_source"
    )

    asset = data_source.add_dataframe_asset(
        name="silver_delivery_asset"
    )

    batch_definition = (
        asset.add_batch_definition_whole_dataframe(
            "silver_dataframe_batch"
        )
    )

    suite = context.suites.add(
        gx.ExpectationSuite(
            name="food_delivery_quality_suite"
        )
    )

    # Completeness checks
    suite.add_expectation(
        gxe.ExpectColumnValuesToNotBeNull(
            column="delivery_id"
        )
    )

    suite.add_expectation(
        gxe.ExpectColumnValuesToNotBeNull(
            column="customer_id"
        )
    )

    # Uniqueness check
    suite.add_expectation(
        gxe.ExpectColumnValuesToBeUnique(
            column="delivery_id"
        )
    )

    # Numerical validity checks
    suite.add_expectation(
        gxe.ExpectColumnValuesToBeBetween(
            column="order_amount",
            min_value=0.01
        )
    )

    suite.add_expectation(
        gxe.ExpectColumnValuesToBeBetween(
            column="delivery_minutes",
            min_value=1
        )
    )

    # Domain consistency checks based on Deliverable 2.
    suite.add_expectation(
        gxe.ExpectColumnValuesToBeInSet(
            column="city_zone",
            value_set=[
                "NORTH",
                "EAST",
                "WEST",
                "SOUTH"
            ]
        )
    )

    suite.add_expectation(
        gxe.ExpectColumnValuesToBeInSet(
            column="delivery_status",
            value_set=[
                "completed",
                "preparing",
                "out for delivery"
            ]
        )
    )

    validation_definition = (
        context.validation_definitions.add(
            gx.ValidationDefinition(
                name="silver_delivery_validation",
                data=batch_definition,
                suite=suite
            )
        )
    )

    checkpoint = context.checkpoints.add(
        gx.Checkpoint(
            name="silver_delivery_checkpoint",
            validation_definitions=[
                validation_definition
            ]
        )
    )

    return checkpoint.run(
        batch_parameters={
            "dataframe": dataframe
        }
    )

## 7. Production Quality Gate

The quality gate converts Great Expectations results into pipeline control.

When validation fails, `QualityGateFailed` is raised. In an Airflow DAG, that exception causes the task to fail and prevents downstream Gold or RAG tasks from running.

In [7]:
class QualityGateFailed(Exception):
    """Raised when Silver data fails Great Expectations validation."""


def enforce_quality_gate(
    validation_result
) -> None:
    """Stop the pipeline when Great Expectations reports failure."""

    if not validation_result.success:
        raise QualityGateFailed(
            "Silver Delta data failed the quality gate. "
            "Downstream stages must not run."
        )

    print("✅ Quality Gate PASSED")
    print(
        "Downstream Gold and RAG stages "
        "are allowed to continue."
    )

## 8. Successful Quality-Gate Run

This section validates the real Silver table produced by Deliverable 2.

The stage emits `START` and `COMPLETE` events when the data passes.

In [8]:
# Convert the small Silver Spark DataFrame into pandas because this
# Great Expectations data source uses an in-memory pandas asset.
silver_pdf = silver_df.toPandas()

quality_run_id = str(
    generate_new_uuid()
)

quality_report_dataset = make_dataset(
    "quality_reports/silver_quality_success",
    namespace="local-quality"
)

emit_lineage_event(
    job_name="validate_silver_quality",
    run_id=quality_run_id,
    state=RunState.START,
    inputs=[silver_dataset],
    outputs=[quality_report_dataset]
)

try:
    validation_result = (
        run_great_expectations_checkpoint(
            silver_pdf
        )
    )

    print(
        "Great Expectations success:",
        validation_result.success
    )

    enforce_quality_gate(
        validation_result
    )

    emit_lineage_event(
        job_name="validate_silver_quality",
        run_id=quality_run_id,
        state=RunState.COMPLETE,
        inputs=[silver_dataset],
        outputs=[quality_report_dataset]
    )

except Exception:
    emit_lineage_event(
        job_name="validate_silver_quality",
        run_id=quality_run_id,
        state=RunState.FAIL,
        inputs=[silver_dataset],
        outputs=[quality_report_dataset]
    )

    raise

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmp87mvbka3' for ephemeral docs site


OpenLineage event emitted: validate_silver_quality → EventType.START


Calculating Metrics:   0%|          | 0/49 [00:00<?, ?it/s]

Great Expectations success: True
✅ Quality Gate PASSED
Downstream Gold and RAG stages are allowed to continue.
OpenLineage event emitted: validate_silver_quality → EventType.COMPLETE


## 9. Intentional Failure-Path Test

The rubric requires proof that the quality gate actually blocks invalid data.

This test creates a bad copy of Silver containing:

- A missing `delivery_id`
- A negative `order_amount`
- An invalid `city_zone`

The expected outcome is:

1. Great Expectations returns `success = False`
2. The gate raises `QualityGateFailed`
3. An OpenLineage `FAIL` event is emitted
4. The downstream marker remains `False`

In [9]:
# Create an intentionally invalid batch without changing the real Silver table.
bad_silver_pdf = silver_pdf.copy()

bad_silver_pdf.loc[
    bad_silver_pdf.index[0],
    "delivery_id"
] = None

bad_silver_pdf.loc[
    bad_silver_pdf.index[1],
    "order_amount"
] = -10.0

bad_silver_pdf.loc[
    bad_silver_pdf.index[2],
    "city_zone"
] = "INVALID_ZONE"

failure_run_id = str(
    generate_new_uuid()
)

bad_dataset = make_dataset(
    "in_memory/bad_silver_test",
    namespace="quality-test"
)

failure_report_dataset = make_dataset(
    "quality_reports/silver_quality_failure",
    namespace="local-quality"
)

downstream_stage_ran = False

emit_lineage_event(
    job_name="validate_bad_silver_quality",
    run_id=failure_run_id,
    state=RunState.START,
    inputs=[bad_dataset],
    outputs=[failure_report_dataset]
)

try:
    bad_validation_result = (
        run_great_expectations_checkpoint(
            bad_silver_pdf
        )
    )

    print(
        "Bad-data Great Expectations success:",
        bad_validation_result.success
    )

    enforce_quality_gate(
        bad_validation_result
    )

    # This line should never run because the gate must raise first.
    downstream_stage_ran = True

    emit_lineage_event(
        job_name="validate_bad_silver_quality",
        run_id=failure_run_id,
        state=RunState.COMPLETE,
        inputs=[bad_dataset],
        outputs=[failure_report_dataset]
    )

except QualityGateFailed as error:
    emit_lineage_event(
        job_name="validate_bad_silver_quality",
        run_id=failure_run_id,
        state=RunState.FAIL,
        inputs=[bad_dataset],
        outputs=[failure_report_dataset]
    )

    print("✅ Expected failure path demonstrated")
    print("Gate message:", error)

print(
    "Did the downstream stage run?",
    downstream_stage_ran
)

assert downstream_stage_ran is False
assert bad_validation_result.success is False

print(
    "✅ Invalid data was blocked before "
    "the downstream stage."
)

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpc3nvwtk1' for ephemeral docs site


OpenLineage event emitted: validate_bad_silver_quality → EventType.START


Calculating Metrics:   0%|          | 0/49 [00:00<?, ?it/s]

Bad-data Great Expectations success: False
OpenLineage event emitted: validate_bad_silver_quality → EventType.FAIL
✅ Expected failure path demonstrated
Gate message: Silver Delta data failed the quality gate. Downstream stages must not run.
Did the downstream stage run? False
✅ Invalid data was blocked before the downstream stage.


## 10. Inspect OpenLineage Events

The generated log provides evidence that real OpenLineage events were emitted.

The expected event types include:

- `START`
- `COMPLETE`
- `FAIL`

In [11]:
import glob
import json
from pathlib import Path

LINEAGE_DIRECTORY = Path("lineage_events")

# Find the main log and any timestamped/rotated OpenLineage files.
lineage_files = sorted(
    set(
        glob.glob(str(LINEAGE_DIRECTORY / "openlineage.log*"))
        + glob.glob(str(LINEAGE_DIRECTORY / "*.json"))
        + glob.glob(str(LINEAGE_DIRECTORY / "*.jsonl*"))
    )
)

print("Lineage files found:")
for file_path in lineage_files:
    print(" -", file_path)

if not lineage_files:
    raise FileNotFoundError(
        "No OpenLineage files were created. "
        "Rerun the successful quality-gate cell and the intentional "
        "failure-path cell before running this verification cell."
    )


def extract_json_objects(file_path):
    """
    Extract OpenLineage JSON events even if objects are:
    - stored one per line;
    - concatenated;
    - separated by commas or other text.
    """

    with open(file_path, "r", encoding="utf-8") as file:
        content = file.read()

    decoder = json.JSONDecoder()
    events = []
    position = 0

    while position < len(content):
        next_object = content.find("{", position)

        if next_object == -1:
            break

        try:
            event, next_position = decoder.raw_decode(
                content,
                next_object
            )

            if isinstance(event, dict):
                events.append(event)

            position = next_position

        except json.JSONDecodeError:
            position = next_object + 1

    return events


lineage_events = []

for file_path in lineage_files:
    lineage_events.extend(
        extract_json_objects(file_path)
    )

print("\nTotal lineage events:", len(lineage_events))

if not lineage_events:
    raise ValueError(
        "Lineage files exist, but no valid JSON events were extracted."
    )

for event in lineage_events:
    print(
        event.get("eventType"),
        "|",
        event.get("job", {}).get("name"),
        "|",
        event.get("eventTime")
    )

event_types = {
    event.get("eventType")
    for event in lineage_events
    if event.get("eventType")
}

print("\nEvent types found:", event_types)

assert "START" in event_types, "No START event was found."
assert "COMPLETE" in event_types, "No COMPLETE event was found."
assert "FAIL" in event_types, "No FAIL event was found."

print(
    "\n✅ START, COMPLETE, and FAIL "
    "OpenLineage events are present."
)

Lineage files found:
 - lineage_events/openlineage.log-20260723-075737.527775.json
 - lineage_events/openlineage.log-20260723-075812.255819.json
 - lineage_events/openlineage.log-20260723-075828.084859.json
 - lineage_events/openlineage.log-20260723-075828.636156.json
 - lineage_events/openlineage.log-20260723-075832.110753.json
 - lineage_events/openlineage.log-20260723-075832.299791.json
 - lineage_events/openlineage.log-20260723-080534.288891.json
 - lineage_events/openlineage.log-20260723-080621.223028.json
 - lineage_events/openlineage.log-20260723-080622.943310.json
 - lineage_events/openlineage.log-20260723-080623.201506.json
 - lineage_events/openlineage.log-20260723-080623.219028.json
 - lineage_events/openlineage.log-20260723-080623.412801.json

Total lineage events: 12
START | load_silver_delta | 2026-07-23T07:57:37.527166+00:00
COMPLETE | load_silver_delta | 2026-07-23T07:58:12.255250+00:00
START | validate_silver_quality | 2026-07-23T07:58:28.084444+00:00
COMPLETE | valida

# Technical Documentation

## Data Flow

```text
Deliverable 2 Silver Delta Table
                ↓
          Load Silver Stage
                ↓
      Great Expectations Suite
                ↓
          Quality Gate
          ↙          ↘
       PASS          FAIL
        ↓             ↓
Downstream allowed   Pipeline stopped
```

## Quality Dimensions

- **Completeness:** required identifiers must not be null
- **Uniqueness:** each delivery ID must occur once
- **Validity:** monetary and duration values must be positive
- **Consistency:** city zones and statuses must use the standardized Silver domains

## Gate Behaviour

`enforce_quality_gate()` raises `QualityGateFailed` when any expectation fails. This is the mechanism that allows Airflow to halt downstream tasks.

## OpenLineage Behaviour

Each demonstrated stage emits structured events with:

- Job namespace and name
- Unique run ID
- Event timestamp
- Input datasets
- Output datasets
- Execution state

The notebook proves all three required states:

- `START`
- `COMPLETE`
- `FAIL`

## Deliverable 2 Dependency

This notebook reads the Silver Delta table at `/content/drive/MyDrive/food_delivery_capstone/delivery_data/silver`. It does not rely on the local `silver_df` variable from another notebook.

For separate Colab sessions, the Delta folder must be persisted to a shared location such as Google Drive.

## Limitations

- The Great Expectations checkpoint uses a pandas asset because the demonstration Silver table is small.
- OpenLineage events are written to a local file rather than sent to a Marquez server.
- Local Colab storage disappears after the runtime ends unless the files are copied to Google Drive.

## Conclusion

This notebook implements a real Great Expectations quality gate over the Silver Delta data and emits real OpenLineage events. It proves both the passing path and the failure path, including downstream blocking.

## Persistent Integration with Deliverable 2

Deliverable 2 and Deliverable 5 remain separate notebooks. Deliverable 2 writes Silver to Google Drive, and Deliverable 5 reads the same persisted Delta path. This avoids Colab's isolated temporary `/content` filesystems.
